In [17]:
from neo4j import GraphDatabase
from dotenv import load_dotenv
import os

load_dotenv()

driver=GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"),os.getenv("NEO4J_PASSWORD"))
)

In [15]:
def test_connection():
    with driver.session() as session:
        session.run("MATCH (n) RETURN n LIMIT 1")

try:
    test_connection()
    print("连接成功")
except Exception as e:
    print("连接失败",e)
finally:
    driver.close()        

连接成功


In [19]:
with driver.session() as session:
    session.run("""
                create (p1:Person{name:'zhangsan',age:30,city:'beijing'}),
                (p2:Person {name:'lisi',age:23,city:'shanghai'})
                """)
    
    session.run("""
                create (c1:Company {name:'shuzhiweilai',industry:'Technology',location:'beijing'}),
                (c2:Company {name:'naixue',industry:'Education',location:'beijing'})
                """)

In [20]:
with driver.session() as session:
    p1=session.run("""
                MATCH (p:Person{name:'zhangsan'})
                set p:Student
                return p
                """)
    p2=session.run("""
                match (p:Person{name:'lisi'})
                set p:Student
                return p
                """)
    print(p1)
    print(p2)

In [21]:
with driver.session() as session:
    session.run("""
                match (p:Person {name:'zhangsan'})
                set p.age=31
                """)

In [22]:
with driver.session() as session:
    session.run("""
                match (p:Person{name:'zhangsan'})
                match (c:Company{name:'shuzhiweilai'})
                create (p)-[:EMPLOYED_BY]->(c)
                """)
    
    session.run("""
                match(p:Person{name:'lisi'})
                match(c:Company{name:'naixue'})
                create (p)-[:EMPLOYED_BY]->(c)
                """)

In [23]:
import re


def neo4j_query_examples(driver,query_type,params=None):
    """
    执行各种Neo4j查询示例
    参数：
    - driver: Neo4j驱动实例
    - query_type: 查询类型，可选值包括：
    'all_persons', 'all_companies', 'filter_by_city', 'all_relationships',
    'specific_relationship', 'node_relationships', 'path_query', 'aggregation',
    'group_by', 'colleagues', 'complex_query', 'param_query', 'subgraph', 'community'
    - params: 查询参数字典，根据查询类型不同而不同
    返回：
    - 查询结果列表
    """
    if params is None:
        params={}
    results=[]
    
    with driver.session() as session:
        if query_type=='all_persons':
            result=session.run("""
            match (p:Person)
            return p.name as name, p.age as age,p.city as city
                               """)
            print("所有person节点：")
            for record in result:
                print(f"姓名：{record['name']},年龄：{record['age']},城市：{record['city']}")
                results.append({
                    'name':record['name'],
                    'age':record['age'],
                    'city':record['city']
                })
        elif query_type=='all_companies':
            result=session.run("""
                               match (c:Company)
                               return c.name as name,c.industry as industry,c.location as location
                               """)
            print("所有company节点")
            for record in result:
                print(f"公司：{record['name']},行业：{record['industry']},位置：{record['location']}")
                results.append({
                    'name':record['name'],
                    'industry':record['industry'],
                    'location':record['location']
                })
        elif query_type=='filter_by_city':
            city=params.get('city','beijing')
            result=session.run("""
                               match (p:Person)
                               where p.city=$city
                               return p.name as name,p.age as age
                               """,{'city':city})
            print(f"{city}人员")
            for record in result:
                print(f"姓名：{record['name']},年龄：{record['age']}")
                results.append({
                    'name':record['name'],
                    'age':record['age']
                })
        elif query_type=='all_relationships':
            result=session.run("""
                               match (p:Person)-[r]->(c:Company)
                               return p.name as person, type(r) as relationship, c.name as company
                               """)
            print("所有人员与公司的关系：")
            for record in result:
                print(f"{record['person']} {record['relationship']} {record['company']}")
                results.append({
                    'person':record['person'],
                    'relationship':record['relationship'],
                    'company':record['company']
                })
        elif query_type=='specific_relationship':
            rel_type=params.get('rel_type','EMPLOYED_BY')
            result=session.run("""
                               match (p:Person)-[:{rel_type}]->(c:Company)
                               return p.name as person, c.name as company
                               """)
            print(f"{rel_type}关系")
            for record in result:
                print(f"{record['person']} 与 {record['company']} 有 {rel_type}关系")
                results.append({
                    'person':record['person'],
                    'company':record['company']
                })
        elif query_type=='node_relationships':
            person_name=params.get('person_name','zhangsan')
            result=session.run("""
                               match (p:Person{name:$name})-[r]->(c)
                               return type(r) as relationship,c.name as connected_to,labels(c) as node_type
                               """,{'name':person_name})
            print(f"{person_name}的所有关系")
            for record in result:
                print(f"关系类型：{record['relationship']},连接到: {record['connected_to']},节点类型：{record['node_type']}")
                results.append({
                    'relationship':record['relationship'],
                    'connected_to':record['connected_to'],
                    'node_type':record['node_type']
                })
        elif query_type=='aggregation':
            result=session.run("""
                               match (p:Person)-[:EMPLOYED_BY]->(c:Company)
                               return c.name as company,count(p) as employee_count,avg(p.age) as avg_age
                               """)
            print("公司员工统计：")
            for record in result:
                print(f"公司：{record['company']},员工数：{record['employee_count']},平均年龄：{round(record['avg_age'],1)}")
                results.append({
                    'company':record['company'],
                    'employee_count':record['employee_count'],
                    'avg_age':record['avg_age'],
                })
        elif query_type=='complex_query':
            min_age=params.get('min_age',25)
            location=params.get('location','beijing')
            result=session.run("""
                               match (p:Person)-[r]->(c:Company)
                               where p.age >$min_age and c.location=$location
                               and (type(r)='EMPLOYED_BY' or type(r)='INVESTED_IN')
                               return p.name as person, p.age as age,
                                type(r) as relationship,c.name as company
                                order by p.age desc
                               """,{'min_age':min_age,'location':location})
            
            print(f"{min_age}岁以上且与{location}公司有雇佣或投资关系的人：")
            for record in result:
                print(f"{record['person']} ({record['age']}岁) {record['relationship']} {record['company']}")
                results.append({
                    'person':record['person'],
                    'age':record['age'],
                    'relationship':record['relationship'],
                    'company':record['company']
                })
        elif query_type=='param_query':
            query_params={
                'min_age':params.get('min_age',25),
                'location':params.get('location','beijing'),
                'relationship_types':params.get('relationship_types',['EMPLOYED_BY','INVESTED_BY'])
            }
            
            result=session.run("""
                               match (p:Person)-[r]->(c:Company)
                               where p.age >$min_age and c.location= $location
                               and type(r) in $relationship_types
                               return p.name as person, type(r) as relationship, c.name as company
                               
                               """,query_params)
            
            print(f"参数化查询结果（年龄》{query_params['min_age']}, 位置{query_params['location']}:")
            for record in result:
                print(f"{record['person']} {record['relationship']} {record['company']}")
                results.append({
                    'person':record['person'],
                    'relationship':record['relationship'],
                    'company':record['company']
                })      
        else:
            print(f"未知的查询类型：{query_type}")
            results.append({'error':f"未知的查询类型：{query_type}"})
    return results

In [24]:
from neo4j import GraphDatabase
from dotenv import load_dotenv
import os

load_dotenv()

driver=GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"),os.getenv("NEO4J_PASSWORD"))
)

try:
    neo4j_query_examples(driver,'all_persons')
    
    neo4j_query_examples(driver,'filter_by_city',{'city':'beijing'})
    neo4j_query_examples(driver,'node_relationships',{'person_name':'zhangsan'})
    neo4j_query_examples(driver,'complex_query',{'min_age':20,'location':'beijing'})
finally:
    driver.close()

所有person节点：
姓名：zhangsan,年龄：31,城市：beijing
姓名：lisi,年龄：23,城市：shanghai
beijing人员
姓名：zhangsan,年龄：31
zhangsan的所有关系
关系类型：EMPLOYED_BY,连接到: shuzhiweilai,节点类型：['Company']
20岁以上且与beijing公司有雇佣或投资关系的人：
zhangsan (31岁) EMPLOYED_BY shuzhiweilai
lisi (23岁) EMPLOYED_BY naixue


In [29]:
import time
import pandas as pd
import concurrent.futures
from neo4j import GraphDatabase

def parallel_batched_import(statement,df,batch_size=100,max_workers=8):
    total=len(df)
    batches=(total+batch_size-1)//batch_size
    start_time=time.time()
    results=[]

    print(f"开始并行导入 {total} 行数据，分为{batches} 个批次，每批{batch_size} 条")

    def process_batch(batch_idx):
        """
    批处理函数，用于处理每个批次的数据
        """

        start=batch_idx*batch_size
        end=min(start+batch_size,total)
        batch=df.iloc[start:end]

        batch_start_time=time.time()
        try:
            with driver.session() as session:
                # UNWIND 是Cypher查询语言中的一个关键字，用于将一个列表展开为多行，$row是一个参数，表示将要传入的行数据
                # 完整意思是 将$row 参数(一个列表)中的每个元素展开，每个元素被赋值给变量value,对每个value执行后续的Cypher语句
                # id name age
                # 0 张三 24
                #1 李四 23
                # 调用.to_dict("record")后，得到
                # [{"id":1,"name":"zhangsan","age":25}]
                result=session.run(
                    "UNWIND $row as value "+statement,row=batch.to_dict("records"))
                summary=result.consume()
                batch_duration=time.time()-batch_start_time

                return {
                    "batch":batch_idx,
                    "rows":end-start,
                    "success":True,
                    "duration":batch_duration,
                    "counters":summary.counters
                }
        except Exception as e:
            batch_duration=time.time()-batch_start_time
            print(f"批次{batch_idx} （行{start}-{end-1}） 处理失败：{str(e)}")

            return {
                    "batch":batch_idx,
                    "rows":end-start,
                    "success":False,
                    "duration":batch_duration,
                    "error":str(e)
                }
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures=[executor.submit(process_batch,i) for i in range(batches)]

        # 处理完成的批次
        for i,future in enumerate(concurrent.futures.as_completed(futures)):
            result=future.result()
            results.append(result)

            if result['success']:
                print(f"批次 {result['batch']} 完成：{result['rows']} 行，耗时：{result['duration']:.2f}秒")
            else:
                print(f"批次 {result['batch']} 失败：{result['rows']} 行，耗时：{result['duration']:.2f}秒，错误：{result.get('error')}")
            # 显示进度
            print(f"进度：{i+1}/{batches} 批次完成 ({((i+1)/batches*100):.1f}%)")

    # 统计结果
    successful_rows=sum(r['rows'] for r in results if r['success'])
    failed_rows=sum(r['rows'] for r in results if not r['success'])

    duration=time.time()-start_time
    rows_per_second=successful_rows/duration if duration >0 else 0

    print(f"导入完成：总计 {total} 行，成功{successful_rows} 行，失败{failed_rows}行")
    print(f"总耗时：{duration:.2f}秒，平均速度:{rows_per_second:.2f}行/秒")
    
    return {
        "total_rows":total,
        "successful_rows":successful_rows,
        "failed_rows":failed_rows,
        "duration_seconds":duration,
        "rows_per_second":rows_per_second,
        "batch_results":results
    }
            

In [30]:
import pandas as pd
from tabulate import tabulate

df_documents = pd.read_parquet('./output/documents.parquet')

# 将数组类型的列转为字符串
df_display = df_documents.copy()
for col in df_display.columns:
    df_display[col] = df_display[col].apply(
        lambda x: str(x) if isinstance(x, (list, tuple)) or (hasattr(x, '__len__') and not isinstance(x, str)) else x
    )

print(tabulate(df_display, headers='keys', tablefmt='pretty', showindex=False, stralign='left', maxcolwidths=[20]*len(df_display.columns)))

+----------------------+-------------------+-----------------+----------------------+----------------------+---------------------+----------+
| id                   | human_readable_id | title           | text                 | text_unit_ids        | creation_date       | raw_data |
+----------------------+-------------------+-----------------+----------------------+----------------------+---------------------+----------+
| 3efa5357dc6c8f0f6262 | 0                 | companyInfo.txt | 华为：从通信设备到智 | ['3755a97bbf1d29f73d | 2026-07-07 20:33:38 |          |
| 609ec584ec8aa64ee766 |                   |                 | 能生态领导者         | a6033a6fc991e50fd5a2 | +0800               |          |
| 4e6f75846bd83bd194d1 |                   |                 | 华为技术有限公司自诞 | ea859a8f37dd7f98fbb4 |                     |          |
| fefc41f657a18ca29f26 |                   |                 | 生于1987年在中国深圳 | 857cceb7f006e1d4c9bd |                     |          |
| 5e5db8d1a8245a02f23e |                

In [31]:
def create_document_nodes(df_documents):
    with driver.session() as session:
        try:
            session.run("create constraint if not exists for (d:__Document__) require d.id is unique")
        except Exception as e:
            print(f"创建约束时出错 (可能已存在): (e)")

    cypher_statement="""
    merge(d:__Document__{id:value.id})
    on create set
        d.human_readable_id=value.human_readable_id,
        d.title=value.title,
        d.text=value.text,
        d.creation_date=value.creation_date,
        d.import_timestamp=timestamp()
    """

    return parallel_batched_import(cypher_statement,df_documents)

In [32]:
from neo4j import GraphDatabase
from dotenv import load_dotenv
import os

load_dotenv()

driver=GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"),os.getenv("NEO4J_PASSWORD"))
)

create_document_nodes(df_documents)

开始并行导入 1 行数据，分为1 个批次，每批100 条
批次 0 完成：1 行，耗时：0.10秒
进度：1/1 批次完成 (100.0%)
导入完成：总计 1 行，成功1 行，失败0行
总耗时：0.10秒，平均速度:9.69行/秒


{'total_rows': 1,
 'successful_rows': 1,
 'failed_rows': 0,
 'duration_seconds': 0.10316348075866699,
 'rows_per_second': 9.693352653923396,
 'batch_results': [{'batch': 0,
   'rows': 1,
   'success': True,
   'duration': 0.10216259956359863,
   'counters': SummaryCounters({'contains-updates': True, 'labels-added': 1, 'nodes-created': 1, 'properties-set': 6})}]}